# US Business Dynamics
### _An Exploratory Data Analysis of Business Growth and Change in the US_

### Project Overview

This project explores patters of US business activity using the Census Bureau's Business Dynamics Statistics dataset. This data provides measures of business activities for the years 1978 through 2023 within various business sectors. 
Its purpose is to lay the data framework that allows me to perform EDA on a large, real-world dataset and demonstrate that process from load to insight in this Jupyter Notebook. 

### Primary Question

**What patterns emerge when running data analysis on the BDS dataset provided by the US Census Bureau?**

### Analysis Guide

This project will progress through:
1. Data acquisition and inspection
2. Data cleaning and quality assessment
3. Feature engineering
4. Univariate exploratory analysis
5. Multivariate exploratory analysis
6. Data visualization
7. Interpretation of key findings
8. Tableau visualization

## 1. Data Acquisition

**Data Source:** Us Census Bureau - Business Dynamics Statistics (BDS) dataset

The BDS provides annual measures of businesss activity in the US including employment, job creation, job desctruction, establishment openings, establishment closings, firm start-ups, and firm shut-downs. The dataset will focus on the State by Sector section of data to provide observations across three insightful dimensions:
- **Time:** Annual observations from 1978 through 2023
- **Geography:** US States
- **Industry:** NAICS industry sectors

Structurally, this allows business activity to be displayed and examined over time, industry, and geographic area, enriching the data with more meaning and depth. 

**Source Links (URLs)**

US Census Bureau Business Dynamics Statistics:
https://www.census.gov/programs-surveys/bds.html

BDS Datasets:
https://www.census.gov/programs-surveys/bds/data.Datasets.html

BDS Codes and Glossary:
https://www.census.gov/programs-surveys/bds/documentation.html

https://www.census.gov/library/reference/code-lists/ansi/ansi-codes-for-states.html

https://www.census.gov/programs-surveys/economic-census/year/2022/guidance/understanding-naics.html

_Import necessary Python packages._

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

Dataset URL

https://www2.census.gov/programs-surveys/bds/tables/time-series/2023/bds2023_st_sec.csv

_Load the raw data in csv form into a pandas dataframe from the data/ folder._

In [3]:
df = pd.read_csv("data/bds2023_st_sec.csv")

_Before running any analysis, we will check the dataset's size (rows, columns)..._

In [4]:
df.shape

(44574, 27)

_...and inspect its headers to identify the types of business metrics available for analysis._

Note, while df.head() is useful to explore a dataset's columns, we do not use it here because pandas cannot display all headers for this specific dataset.

In [5]:
df.columns.tolist()

['year',
 'st',
 'sector',
 'firms',
 'estabs',
 'emp',
 'denom',
 'estabs_entry',
 'estabs_entry_rate',
 'estabs_exit',
 'estabs_exit_rate',
 'job_creation',
 'job_creation_births',
 'job_creation_continuers',
 'job_creation_rate_births',
 'job_creation_rate',
 'job_destruction',
 'job_destruction_deaths',
 'job_destruction_continuers',
 'job_destruction_rate_deaths',
 'job_destruction_rate',
 'net_job_creation',
 'net_job_creation_rate',
 'reallocation_rate',
 'firmdeath_firms',
 'firmdeath_estabs',
 'firmdeath_emp']

_We will inspect (.info()) the dataset's structure, data types and non-null counts to determine how each variable is interpreted during import and to identify fields that may require more processing._

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 44574 entries, 0 to 44573
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   year                         44574 non-null  int64
 1   st                           44574 non-null  int64
 2   sector                       44574 non-null  str  
 3   firms                        44574 non-null  str  
 4   estabs                       44574 non-null  str  
 5   emp                          44574 non-null  str  
 6   denom                        44574 non-null  str  
 7   estabs_entry                 44574 non-null  str  
 8   estabs_entry_rate            44574 non-null  str  
 9   estabs_exit                  44574 non-null  str  
 10  estabs_exit_rate             44574 non-null  str  
 11  job_creation                 44574 non-null  str  
 12  job_creation_births          44574 non-null  str  
 13  job_creation_continuers      44574 non-null  str  
 14  j

Observation:

The first three columns in our dataframe represent yea, state, and sector. These do not need to be converted to numeric form as we will convert them to their respective indicators later as a cleaning step. Furthermore, most of the raw BDS data represents numeric counts or rates (int64), but pandas imported 25 of the 27 columns as strings (str), which may hamper our data analysis. Before we convert these strings to numbers, we will investigate further to see why these values are seen as strings.

Assumption:

We can assume the column header `firms` should represent subsequent data as the numeric value of firms. 

Action:

Since we have an exorbitant amount of data to sift through, we will first see a light representation of how often a str type value appears rather than int64.

_We will display the first 20 value counts of the column labeled `firms` in our dataset without dropping any NaN values._

In [7]:
df["firms"].value_counts(dropna=False).head(20)

firms
39     49
73     47
74     47
163    44
44     44
78     42
103    42
D      42
125    41
60     41
90     41
105    41
131    41
69     40
89     40
104    40
48     40
85     40
93     39
161    39
Name: count, dtype: int64

_Then, we will create a temporary dataframe from the original while converting all data types to int64 (.to_numeric), force all non int64 values into NaN values (errors=coerce), as well as sift through the numeric data for anomalies to show us why this column may be imported as str type and how often those anomalies occur._

In [8]:
firms_numeric = pd.to_numeric(df["firms"], errors="coerce")

df.loc[firms_numeric.isna(), "firms"].value_counts(dropna=False)

firms
D    42
Name: count, dtype: int64

The output of the cell above lists "D" as the culprit as to why our data is of str type and not numeric. In perusing the BDS Glossary, we can see:
> Disclosure Suppression – Disclosure suppressions are made when a cell has too few firms. Cells suppressed due to containing too few firms will appear as “D”.

It is always a good idea to take into consideration auxiliary supporting documentation when assessing datasets for meaning.

All values in the `firms` column designated as "D" are therefore **NOT** zeros, rather they are suppressed and cannot be inferred through data analysis. We will check to see if any other column within our dataframe that is of data type str contains indicators we can look up in our glossary.

Anomaly scrounging:

_We will start with an empty dictionary, we will convert all columns in our dataframe to numeric starting with the fourth column (df.columns[3:]) as we know the first three don't necessarily need numeric conversion, identify which ones cannot convert, load those values into the empty dictionary and then we will display its contents._

In [9]:
BDS_indicators = {}

for column in df.columns[3:]:
    numeric_version = pd.to_numeric(df[column], errors="coerce")

    invalid_values = (
        df.loc[numeric_version.isna(), column]
        .value_counts()
        .to_dict()
    )

    if invalid_values:
        BDS_indicators[column] = invalid_values

BDS_indicators

{'firms': {'D': 42},
 'estabs': {'D': 42},
 'emp': {'D': 42},
 'denom': {'D': 43},
 'estabs_entry': {'D': 445},
 'estabs_entry_rate': {'D': 445, 'N': 1},
 'estabs_exit': {'D': 542},
 'estabs_exit_rate': {'D': 542, 'N': 1},
 'job_creation': {'D': 42},
 'job_creation_births': {'D': 445},
 'job_creation_continuers': {'D': 51},
 'job_creation_rate_births': {'D': 445, 'N': 1},
 'job_creation_rate': {'D': 42, 'N': 1},
 'job_destruction': {'D': 43},
 'job_destruction_deaths': {'D': 542},
 'job_destruction_continuers': {'D': 51},
 'job_destruction_rate_deaths': {'D': 542, 'N': 1},
 'job_destruction_rate': {'D': 43, 'N': 1},
 'net_job_creation': {'D': 43},
 'net_job_creation_rate': {'D': 43, 'N': 1},
 'reallocation_rate': {'D': 42, 'N': 1},
 'firmdeath_firms': {'D': 1339},
 'firmdeath_estabs': {'D': 1339},
 'firmdeath_emp': {'D': 1339}}

In looking through the data ouput, we see that str values "D" and "N" appear frequently. From the glossary:
> Rate Not Available – Rates that cannot be calculated due to a denominator of ‘0’ will appear as “N”.

These indicators are also not evenly distributed through the dataset, with some columns containing more or less than others of these indicators. We, again, will **NOT** consider these values as zeros. We will take a look at these later to see if we can infer anything about how often we see them.

As of now, our dataset is ready to clean.

## 2. Data Cleaning

In the previous exercise, we found that many values were listed as str type because they represented unavailable data and utilized an alphabetic indicator code instead of a numeric value. To progress with our exploratory data analysis, we should find a way to convert these values into data types that can be analyzed.

_We will go forward with a copy of the dataframe so as to not irreversably alter the original. We will further sequester the columns to convert into numeric form by creating a new variable containing those columns and disregarding the first three (df_clean.columns[3:])._

In [10]:
df_clean = df.copy()

In [11]:
columns_tonumeric = df_clean.columns[3:]

_Now, we will use the variable and undergo conversion from whatever datatype they currently exist as to numeric. All errors (alphabetic values/strings) will be coerced into values of NaN._

In [12]:
for column in columns_tonumeric:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    )

_Let's check that the conversion worked by listing the datatypes contained within our copied dataframe._

In [13]:
df_clean.info()

<class 'pandas.DataFrame'>
RangeIndex: 44574 entries, 0 to 44573
Data columns (total 27 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   year                         44574 non-null  int64  
 1   st                           44574 non-null  int64  
 2   sector                       44574 non-null  str    
 3   firms                        44532 non-null  float64
 4   estabs                       44532 non-null  float64
 5   emp                          44532 non-null  float64
 6   denom                        44531 non-null  float64
 7   estabs_entry                 44129 non-null  float64
 8   estabs_entry_rate            44128 non-null  float64
 9   estabs_exit                  44032 non-null  float64
 10  estabs_exit_rate             44031 non-null  float64
 11  job_creation                 44532 non-null  float64
 12  job_creation_births          44129 non-null  float64
 13  job_creation_continuers    

From the logic check above, we can see that the columns we identified as needing conversion to numeric data type have all become floats which are computable. We still need to deal with the columns `st` and `sector`. 

In the explanatory links to the BDS coding above, we can find mappings for the numeric FIPS and NAICS codes contained in our columns representing state and sector. 

_We will map the state codes to state names._

In [14]:
state_codes = {
    1: "Alabama",
    2: "Alaska",
    4: "Arizona",
    5: "Arkansas",
    6: "California",
    8: "Colorado",
    9: "Connecticut",
    10: "Delaware",
    11: "District of Columbia",
    12: "Florida",
    13: "Georgia",
    15: "Hawaii",
    16: "Idaho",
    17: "Illinois",
    18: "Indiana",
    19: "Iowa",
    20: "Kansas",
    21: "Kentucky",
    22: "Louisiana",
    23: "Maine",
    24: "Maryland",
    25: "Massachusetts",
    26: "Michigan",
    27: "Minnesota",
    28: "Mississippi",
    29: "Missouri",
    30: "Montana",
    31: "Nebraska",
    32: "Nevada",
    33: "New Hampshire",
    34: "New Jersey",
    35: "New Mexico",
    36: "New York",
    37: "North Carolina",
    38: "North Dakota",
    39: "Ohio",
    40: "Oklahoma",
    41: "Oregon",
    42: "Pennsylvania",
    44: "Rhode Island",
    45: "South Carolina",
    46: "South Dakota",
    47: "Tennessee",
    48: "Texas",
    49: "Utah",
    50: "Vermont",
    51: "Virginia",
    53: "Washington",
    54: "West Virginia",
    55: "Wisconsin",
    56: "Wyoming"
}

In [15]:
df_clean["st"] = df_clean["st"].map(state_codes)

_Next, we will do the same for mapping the sector codes to individually identifiable sectors._

In [16]:
sector_codes = {
    "11": "Agriculture, Forestry, Fishing and Hunting",
    "21": "Mining, Quarrying, and Oil and Gas Extraction",
    "22": "Utilities",
    "23": "Construction",
    "31-33": "Manufacturing",
    "42": "Wholesale Trade",
    "44-45": "Retail Trade",
    "48-49": "Transportation and Warehousing",
    "51": "Information",
    "52": "Finance and Insurance",
    "53": "Real Estate and Rental and Leasing",
    "54": "Professional, Scientific, and Technical Services",
    "55": "Management of Companies and Enterprises",
    "56": "Administrative and Support and Waste Management and Remediation Services",
    "61": "Educational Services",
    "62": "Health Care and Social Assistance",
    "71": "Arts, Entertainment, and Recreation",
    "72": "Accommodation and Food Services",
    "81": "Other Services (except Public Administration)"
}

In [17]:
df_clean["sector"] = df_clean["sector"].map(sector_codes)

_Let's now validate that the mappings have occured and that we have not inadvertently created any NaN values by letting values that were not included in the code listings through the cracks._

In [18]:
df_clean[["year", "st","sector"]].head(2)

,year,st,sector
0,1978,Alabama,"Agriculture, Forestry, Fishing and Hunting"
1,1978,Alabama,"Mining, Quarrying, and Oil and Gas Extraction"


In [19]:
print(df_clean["st"].isna().sum())

print(df_clean["sector"].isna().sum())

0
0


We can see that the mapping has occurred successfully, and that there are zero Nan values for both the `st` and `sector` column values. We can check the `year` column to figure out if there are any missing years. This will prevent any confounding observations when undergoing analysis in the near future.

_We will check for year continuity by subtracting the difference between the maximum and minimum listed year in the column's values and adding one value to insure we are counting all years in that range (+ 1) from the amount of unique years (["year"].unique())._

In [20]:
expected_years = set(range(
    df_clean["year"].min(),
    df_clean["year"].max() + 1
))

actual_years = set(df_clean["year"].unique())

expected_years - actual_years

set()

This yields an expty set, which we can infer means there are no missing values betwee the minimum and maximum listed years in our dataframe. 

Now we can search for missing rows and make sure all rows are unique. This gives us assurance that there are no duplicate rows. 

_To search for missing rows, we count the unique amount of years, states, and sectors in our data as a total dimension, and then compare that with our current row count._

In [21]:
n_years = df_clean["year"].nunique()
n_states = df_clean["st"].nunique()
n_sectors = df_clean["sector"].nunique()

print("Years = " + str(n_years))
print("States = " + str(n_states))
print("Sectors = " + str(n_sectors))
print("Dimension = " + str(n_years * n_states * n_sectors))

Years = 46
States = 51
Sectors = 19
Dimension = 44574


In [22]:
row_dimension = n_years * n_states * n_sectors
row_actual = len(df_clean)

print("Expected rows = " + str(row_dimension))
print("Actual rows = " + str(row_actual))

Expected rows = 44574
Actual rows = 44574


While this does affirm our idea of row counts matching, it does not tell us if any of the rows are duplicates. To find out if our data contains duplicates, we can use pandas' native method (.duplicated()).

_We will use the method to find duplicates within the dimension defined above (`year` * `st` * `sector`). A sum of these duplicates should equal zero._

In [23]:
duplicate_count = df_clean.duplicated(
    subset=["year", "st", "sector"]
).sum()

duplicate_count

np.int64(0)

A result of zero shows that our dataset has no duplicates.

**Duplicate rows are a common occurence in datasets that will undergo analysis. We have escaped tragedy here by finding none, but had we found any, we could have used pandas' native duplicate dropping method (.drop_duplicates()) to remove them and continue forward.**

_Let's display a visual representation of all the value counts and dimensions we have validated thus far for reference._

In [24]:
print(f"Years: {n_years}")
print(f"States: {n_states}")
print(f"Sectors: {n_sectors}")
print(f"Expected rows: {row_dimension:,}")
print(f"Actual rows: {row_actual:,}")
print(f"Duplicate Year x State x Sector combinations: {duplicate_count}")

Years: 46
States: 51
Sectors: 19
Expected rows: 44,574
Actual rows: 44,574
Duplicate Year x State x Sector combinations: 0


The last part of this is a quality assessment on our data. Tis pertains a lot to the logic behind the numbers, and brings synthesis of the column headers to the forefront. Knowing what they truly represent allow us to make sure we are working with good data. Everyone who's taken any data course this century knows the adage _"Garbage In, Garbage Out"_ so these next few steps affirm that our datacontains no garbage. 

Using the list of column headers above, we will comprehend which values could not possibly summate to be negative and assure ourselves that they have positive counts. 

_We first collect the columns we believe should be non-negative into a list [], and then check that list for any negative values in those columns by summing their total negative values. If no negative values exist, then the summation will yield zero as a result._

In [25]:
nonnegative_columns = [
    "firms",
    "estabs",
    "emp",
    "denom",
    "estabs_entry",
    "estabs_exit",
    "job_creation",
    "job_creation_births",
    "job_creation_continuers",
    "job_destruction",
    "job_destruction_deaths",
    "job_destruction_continuers",
    "firmdeath_firms",
    "firmdeath_estabs",
    "firmdeath_emp"
]

(df_clean[nonnegative_columns] < 0).sum()

firms                         0
estabs                        0
emp                           0
denom                         0
estabs_entry                  0
estabs_exit                   0
job_creation                  0
job_creation_births           0
job_creation_continuers       0
job_destruction               0
job_destruction_deaths        0
job_destruction_continuers    0
firmdeath_firms               0
firmdeath_estabs              0
firmdeath_emp                 0
dtype: int64

Observation:

Seeing that we dont have negative values in the rows that we belive won't have negative values, that seems like a win for continuity and quality. One more facet of quality within the realm of logic exists as an equation we can surmise form looking at the column headers again. We have `net_job_creation`, `job_creation`, and `job_destruction`. Logically, we should be able to create an equation from these three columns that would describe their relationship.

Assumption:

net_job_creation = job_creation - job_destruction

Action:

Let's calculate this relationship and see what we get. 

_Now, we can create a new variable called net_job_validation using the aforementioned columns and drop any missing values while showing the total amount of values we are dealing with in this relationship._

In [27]:
net_job_validation = df_clean[
    [
        "job_creation",
        "job_destruction",
        "net_job_creation"
    ]
].dropna()

len(net_job_validation)

44531

_Let's execute the equation of this relationship by subtracting jobs lost (["job_destruction"]) from the jobs that were created (["job_creation"]) and compare/equate that (.eq()) with the amount of values (.value_counts()) we validated in the previous step to show any inconsistencies and their amount._

In [28]:
net_job_equation = (
    net_job_validation["job_creation"]
    - net_job_validation["job_destruction"]
)

net_job_validation["net_job_creation"].eq(
    net_job_equation
).value_counts()

True     44521
False       10
Name: count, dtype: int64

The ideal result here would have been: 
> True     44531

but we seem to have 10 discrepancies (False) in our data. That means net_job_creation =/= job_creation - job_desctruction for 10 rows of our data. Let's dig a little deeper and figure out if these discrepancies are meaningful.

_To sequester our quality check here, we will create a new variable as we did with net_job_equation above called mismatches. We will use this to match indeces with our df_clean dataframe and create a copy to have a fresh dataframe to work with. Then we can append the new dataframe with our actual discrepancy values in a new column called `difference` and display our assessment._

In [ ]:
mismatches = net_job_validation[
    ~net_job_validation["net_job_creation"].eq(
        net_job_equation
    )
]

net_job_mismatches = df_clean.loc[
    df_clean.index.isin(mismatches.index),
    [
        "year",
        "st",
        "sector",
        "job_creation",
        "job_destruction",
        "net_job_creation"
    ]
].copy()

net_job_mismatches["net_job_equation"] = (
    net_job_mismatches["job_creation"]
    - net_job_mismatches["job_destruction"]
)

net_job_mismatches["difference"] = (
    net_job_mismatches["net_job_creation"]
    - net_job_mismatches["net_job_equation"]
)

net_job_mismatches

net_job_mismatches["difference"].value_counts().sort_index()

difference
-1.0    6
 1.0    4
Name: count, dtype: int64

What this output tells us is that of 10 total discrepancies to our assumed equation, 6 are off by -1 and 4 are off by +1. The percentage of these discrepancies will inform the meaning behind them. We have 44531 values being looked at here, and of 10 discrepancies, each one of those is +/-1. 
> 1 / 44531 = 0.000022456... = 0.0022456...%

That's a low percentage and yields a low meanginfulness to these discrepancies, so we will leave them be for now.

Before we conclude these data cleaning aspect of this exercise, we will look again at our "D" and "N" Census indicators. Our BDS_inidcators variable above shows us the frequency with which we see these values, but does not show where in our dataset they may occur most often. Knowing this could pave the path to EDA with more insight and explain things better for us later on.

_We will select the columns `year`, `sector`, and `st` and within each of these, we will find individual row counts for NaN values, summate them, and sort the top ten rows in descending order for textual assessment._

In [36]:
df_clean[columns_tonumeric].isna().sum(axis=1).groupby(
    df_clean["year"]
).sum().sort_values(ascending=False).head(10)

year
2020    257
2018    249
2016    236
2019    235
1984    229
1979    224
2012    222
1995    221
2013    220
2010    218
dtype: int64

In [34]:
df_clean[columns_tonumeric].isna().sum(axis=1).groupby(
    df_clean["sector"]
).sum().sort_values(ascending=False).head(10)

sector
Utilities                                                                   4065
Mining, Quarrying, and Oil and Gas Extraction                               2484
Management of Companies and Enterprises                                      993
Agriculture, Forestry, Fishing and Hunting                                   974
Educational Services                                                          22
Manufacturing                                                                  4
Accommodation and Food Services                                                0
Administrative and Support and Waste Management and Remediation Services       0
Construction                                                                   0
Finance and Insurance                                                          0
dtype: int64

In [35]:
df_clean[columns_tonumeric].isna().sum(axis=1).groupby(
    df_clean["st"]
).sum().sort_values(ascending=False).head(10)

st
District of Columbia    1400
Delaware                 682
Rhode Island             646
Hawaii                   545
Vermont                  513
South Dakota             434
Maine                    411
New Hampshire            365
North Dakota             357
Alaska                   257
dtype: int64

For `year`, we simply don't see much concentration in missing values in the output. They are interspersed among our rows and don't tell us much in that regard.

For `sector`, things begin to get interesting. We can see a large difference between the first row in this output and the second, third, and fourth in descending fashion. This tells us that the first row, "Utilities", has around double the NaN values that "Mining, Quarrying, and Oil and Gas Extraction" has. When running EDA, we need to remember that these sectors could give us wonky results as compared to the others.

In the `st` output, we see a similar disparity in NaN frequency with DC leading by over double the next-highest value count in Delaware. Good to note these findings and we don't need to act on them just yet, if at all.

Our dataset is now ready for feature engineering.

## 3. Feature Engineering

This is our chance to increase the interpretability of our dataset by creating entirely new variables we can pull from during EDA. 

Observation:

Within the list of column headers of this dataset, we can find many pre-existing variables like `emp` (number of employees) and `estab_per_firm` (establishments per firm). These numbers and rates tell us a lot about the data and will be included in our EDA. One feature that is not included is `decade`, basically a grouping of high-level data by decade. 

Assumption:

Business in the 1980s changes differently than in the 2010s. 

Action:

Let's create the `decade` feature for this dataset. Basically, any year value between 1980 and 1989 will be dubbed 1980s and so on. The 1970s and 2020s pose an issue with normalization in that within this dataset they contain a year amount that does not equal 10. Normally (see what I did there), I would *normalize* these features, but my previous coursework has the normalization step as needed during analysis, so I will trust that as the process to use going forward. 

_To engineer this feature, we will take each value in column year (df_clean["year"]), floor divide it by 10 (// 10) which divides the value by a predefined number and rounds down, multiply that value by 10 which appends a zero at the end of it, convert that value to a string (.astype(str)), and concatenate an "s" to the end that new string._

In [44]:
df_clean["decade"] = ((df_clean["year"] // 10) * 10).astype(str) + "s"

We can validate that the feature has been engineered correctly in two ways:

_First, we will display the `year` and newly created `decade` columns from our dataframe, drop any duplicate values from these columns to make the output consist of unique values, and display 15 rows to give us a quick view of what we've created._

In [45]:
df_clean[["year", "decade"]].drop_duplicates().head(15)

,year,decade
0,1978,1970s
969,1979,1970s
1938,1980,1980s
2907,1981,1980s
3876,1982,1980s
4845,1983,1980s
5814,1984,1980s
6783,1985,1980s
7752,1986,1980s
8721,1987,1980s


_Or, we can validate the feature by seeing if the expected values within each decade (.groupby("decade")) add up to the amount of distinct year values (.nunique()) included in this dataset, hence why we did not normalize these data._

In [46]:
df_clean.groupby("decade")["year"].nunique()

decade
1970s     2
1980s    10
1990s    10
2000s    10
2010s    10
2020s     4
Name: year, dtype: int64

This all makes sense, as the 1970s contains only the years 1978 and 1979 and the 2020s contains only the years between 2020 and 2023. These partial decades will not be treated as equivalent to the decades that contain 10 distinct years when aggregating values. Where appropriate, normalization can occur when comparing multiple decades that include these columns. 

This concludes feature engineering and prepares us for univariate exploratory analysis. 